# Phase 4 CJPE: LoRA, LoRMA, full fine-tuning and distillation on my phase 1 gqa_swa model

Attach the phase 1 output (`gqa_swa_best.pt` and `bpe_legal_16k_v2.pkl`) and add the `HF` secret.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, gc, re, json, math, time, glob, pickle, random, zipfile
from pathlib import Path
from collections import Counter
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt
from huggingface_hub import login, hf_hub_download
from sklearn.metrics import precision_recall_fscore_support, classification_report, f1_score
from IPython.display import FileLink, display


# timestamped and flushed, so the kaggle console keeps updating
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for secret_name in ("HF", "HF_TOKEN"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(secret_name)
            if HF_TOKEN:
                break
        except Exception:
            pass
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    log("Logged in.")
else:
    log("WARNING: no HF token found, trying an unauthenticated download")


## Config

In [ ]:
# SMOKE = tiny run of everything on a few cases, RUN_PREFLIGHT = quick forward / backward / zip check before training
SMOKE = False
RUN_PREFLIGHT = True
# print a progress line every N batches
LOG_EVERY = 100
SEED = 42

# docs are cut into 512 token windows with a stride of 64
CHUNK_LEN = 512
STRIDE = 64
# train keeps the head and the tail of the case, dev and test get a few more windows
MAX_CHUNKS_TRAIN = 2
MAX_CHUNKS_EVAL = 4
BATCH_SIZE = 16
# effective batch is 32 cases per optimizer step
GRAD_ACCUM = 2
# how many chunks go through the backbone at once
CHUNK_MICRO_BATCH = 32

WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.10
LABEL_SMOOTHING = 0.03
CLIP_NORM = 1.0

RANK, ALPHA = 8, 16.0
LORA_DROPOUT = 0.05
STUDENT_LAYERS = 8
KD_ALPHA = 0.4
KD_TEMP = 2.0
EXPERIMENTS = [
    {"num": 1, "method": "lora", "epochs": 2, "backbone_lr": 3e-4, "head_lr": 5e-4},
    {"num": 2, "method": "lorma", "epochs": 2, "backbone_lr": 3e-4, "head_lr": 5e-4},
    {"num": 3, "method": "full", "epochs": 2, "backbone_lr": 2e-5, "head_lr": 4e-4},
    {"num": 4, "method": "distillation", "epochs": 2, "backbone_lr": 2e-4, "head_lr": 5e-4, "kd": True},
]

# None = look for them in /kaggle/input
CKPT_PATH = None
TOK_PATH = None

# not stored in the checkpoint, same as phase 1
MAX_SEQ_LEN = 1023
ATTN_WINDOW, GLOBAL_EVERY = 256, 4

WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "cjpe_experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = dev.type == "cuda"
log(f"device: {dev} | {torch.cuda.get_device_name(0) if use_amp else 'cpu'} | SMOKE={SMOKE}")


## Phase 1 checkpoint and tokenizer

In [ ]:
# take the checkpoint with the highest step if there are several
if CKPT_PATH is None:
    hits = glob.glob("/kaggle/input/**/gqa_swa_best.pt", recursive=True)
    print("checkpoints found:", hits)
    if hits:
        CKPT_PATH = hits[0] if len(hits) == 1 else max(hits, key=lambda p: torch.load(p, map_location="cpu", weights_only=False).get("step", 0))
assert CKPT_PATH and os.path.isfile(CKPT_PATH), "gqa_swa_best.pt not found in /kaggle/input"

# tokenizer that sits closest to the checkpoint in the folder tree
if TOK_PATH is None:
    toks = glob.glob("/kaggle/input/**/bpe_legal_16k_v2.pkl", recursive=True)
    print("tokenizers found:", toks)
    if toks:
        TOK_PATH = max(toks, key=lambda t: len(os.path.commonpath([CKPT_PATH, t])))
assert TOK_PATH and os.path.isfile(TOK_PATH), "bpe_legal_16k_v2.pkl not found in /kaggle/input"
print("checkpoint:", CKPT_PATH)
print("tokenizer :", TOK_PATH)


class CustomBPETokenizer:
    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>"]
        self.pad_token_id = 0
        self.unk_token_id = 1
        self.bos_token_id = 2
        self.eos_token_id = 3
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\S|\s+(?!\S)|\s+""")
        self.merges = {}
        self.vocab = {}
        self.inverse_vocab = {}
        self._word_cache = {}

    def _tokenize_word(self, word):
        word_tuple = tuple(word)
        if len(word_tuple) <= 1:
            return list(word_tuple)
        while True:
            pairs = [(word_tuple[i], word_tuple[i + 1]) for i in range(len(word_tuple) - 1)]
            valid_merges = [p for p in pairs if p in self.merges]
            if not valid_merges:
                break
            best_pair = min(valid_merges, key=lambda p: self.vocab[self.merges[p]])
            first, second = best_pair
            merged_tok = self.merges[best_pair]
            new_tuple = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == first and word_tuple[i + 1] == second:
                    new_tuple.append(merged_tok)
                    i += 2
                else:
                    new_tuple.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_tuple)
            if len(word_tuple) == 1:
                break
        return list(word_tuple)

    def encode(self, text, add_special_tokens=False):
        if not text:
            return []
        cache = self._word_cache
        ids = []
        for word in self.pat.findall(text):
            word_ids = cache.get(word)
            if word_ids is None:
                word_ids = [self.vocab.get(tok, self.unk_token_id) for tok in self._tokenize_word(word)]
                if len(cache) < 2_000_000:
                    cache[word] = word_ids
            ids.extend(word_ids)
        if add_special_tokens:
            ids = [self.bos_token_id] + ids + [self.eos_token_id]
        return ids

    @property
    def vocab_size(self):
        return len(self.vocab)

    @classmethod
    def load(cls, filepath):
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        tok = cls(vocab_size=data["target_vocab_size"])
        tok.vocab = data["vocab"]
        tok.inverse_vocab = data["inverse_vocab"]
        tok.merges = data["merges"]
        tok.special_tokens = data["special_tokens"]
        print(f"Loaded tokenizer from {filepath} (Vocab size: {tok.vocab_size})")
        return tok


tokenizer = CustomBPETokenizer.load(TOK_PATH)
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID, UNK_ID, BOS_ID, EOS_ID = tokenizer.pad_token_id, tokenizer.unk_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id


## Data

In [ ]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P4 = "Phase_4_CJPE"


def fetch(rel):
    log(f"fetching {rel}")
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)


def read_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


# reads one split into a list of {id, text, label}, the json layout is not always the same
def load_cjpe_split(rel_path, is_test=False):
    raw = read_json(fetch(rel_path))

    if isinstance(raw, dict):
        if len(raw) == 1 and isinstance(next(iter(raw.values())), list) and next(iter(raw.keys())) in ("data", "examples", "cases", "items"):
            raw_list = next(iter(raw.values()))
        elif "text" in raw and ("id" in raw or "doc_id" in raw or "case_id" in raw):
            raw_list = [raw]
        else:
            raw_list = []
            for k, v in raw.items():
                if isinstance(v, dict):
                    item = dict(v)
                    item.setdefault("id", str(k))
                    raw_list.append(item)
                elif isinstance(v, list):
                    raw_list.append({"id": str(k), "text": " ".join(str(x) for x in v)})
                elif isinstance(v, str):
                    raw_list.append({"id": str(k), "text": v})
    elif isinstance(raw, list):
        raw_list = raw
    else:
        raise ValueError(f"Unsupported JSON root type in {rel_path}: {type(raw)}")

    records = []
    for idx, item in enumerate(raw_list):
        if not isinstance(item, dict):
            continue
        cid = str(item.get("id", item.get("case_id", item.get("doc_id", item.get("uid", f"case_{idx}")))))
        text = item.get("text", item.get("document", item.get("judgment", item.get("facts", item.get("sentences", "")))))
        if isinstance(text, list):
            text = " ".join(str(s) for s in text)
        text = str(text).strip()

        label = None
        if not is_test:
            for lk in ("label", "pred", "target", "decision", "outcome", "y"):
                if lk in item and item[lk] is not None:
                    label = int(item[lk])
                    break
            assert label in (0, 1), f"Invalid or missing binary label {label!r} for case {cid} in {rel_path}"

        records.append({"id": cid, "text": text, "label": label})

    assert len(records) > 0, f"loaded 0 records from {rel_path}"
    return records


train_data = load_cjpe_split(f"{P4}/Train/train.json")
dev_data = load_cjpe_split(f"{P4}/Train/dev.json")
# test has no labels
test_data = load_cjpe_split(f"{P4}/Test/test.json", is_test=True)

words_per_case = np.array([len(x["text"].split()) for x in train_data])
label_counts = Counter(x["label"] for x in train_data)
dev_counts = Counter(x["label"] for x in dev_data)
n0, n1 = label_counts.get(0, 1), label_counts.get(1, 1)
total_train = n0 + n1

# class weights for the weighted ce loss, the rarer label weighs more
w0 = total_train / (2.0 * n0)
w1 = total_train / (2.0 * n1)
CLASS_WEIGHTS = torch.tensor([w0, w1], dtype=torch.float32, device=dev)

print(f"train: {len(train_data)} cases | label 0 -> {n0} ({n0 / total_train:.1%}), label 1 -> {n1} ({n1 / total_train:.1%})")
print(f"dev  : {len(dev_data)} cases | label 0 -> {dev_counts[0]}, label 1 -> {dev_counts[1]}")
print(f"test : {len(test_data)} cases | sample ids: {[x['id'] for x in test_data[:3]]}")
print(f"words/case (train): mean {words_per_case.mean():.0f}, median {np.median(words_per_case):.0f}, p95 {np.percentile(words_per_case, 95):.0f}, max {words_per_case.max()}")
print(f"class weights: w0={w0:.4f}, w1={w1:.4f}")


## Chunked windows

In [ ]:
# cuts a case into 512 token windows (stride 64) and keeps the head and tail windows
def fast_encode_and_chunk(text, chunk_len=CHUNK_LEN, stride=STRIDE, max_chunks=MAX_CHUNKS_TRAIN):
    # very long cases are clipped to their first and last words before bpe, the middle gets dropped anyway
    words = text.split()
    max_words_needed = max_chunks * 500
    if len(words) > max_words_needed * 2:
        half_w = max_words_needed
        head_ids = tokenizer.encode(" ".join(words[:half_w]), add_special_tokens=False)
        tail_ids = tokenizer.encode(" ".join(words[-half_w:]), add_special_tokens=False)
        token_ids = head_ids + tail_ids
    else:
        token_ids = tokenizer.encode(text, add_special_tokens=False)

    # room for bos and eos
    eff_len = chunk_len - 2
    if not token_ids:
        token_ids = [UNK_ID]

    raw_windows = []
    n = len(token_ids)
    if n <= eff_len:
        raw_windows.append(token_ids)
    else:
        step = max(1, chunk_len - stride)
        for start in range(0, n, step):
            win = token_ids[start: start + eff_len]
            raw_windows.append(win)
            if start + eff_len >= n:
                break

    # first half of the windows from the head (facts), the rest from the tail (reasoning / verdict)
    if max_chunks is not None and len(raw_windows) > max_chunks:
        head_k = max(1, max_chunks // 2)
        tail_k = max_chunks - head_k
        raw_windows = raw_windows[:head_k] + raw_windows[-tail_k:]

    chunks, lengths = [], []
    for win in raw_windows:
        seq = [BOS_ID] + win + [EOS_ID]
        L = len(seq)
        if L < chunk_len:
            seq = seq + [PAD_ID] * (chunk_len - L)
        chunks.append(seq)
        lengths.append(L)

    return np.asarray(chunks, dtype=np.int16), np.asarray(lengths, dtype=np.int16)


# tokenize every case once and keep the chunks in ram
def build_split_cache(records, max_chunks, split_name):
    t0 = time.time()
    cached = []
    total = len(records)
    for i, rec in enumerate(records, 1):
        chunks, lengths = fast_encode_and_chunk(rec["text"], CHUNK_LEN, STRIDE, max_chunks)
        cached.append({
            "id": rec["id"],
            "chunks": chunks,
            "lengths": lengths,
            "label": -1 if rec["label"] is None else int(rec["label"]),
        })
        if i % 5000 == 0 or i == total:
            log(f"  [{split_name}] tokenized {i}/{total} cases ({i / total:.0%}) in {time.time() - t0:.1f}s")
    return cached


if SMOKE:
    log("SMOKE=True: 128 train, 64 dev and the full test set")
    active_train_records = train_data[:128]
    active_dev_records = dev_data[:64]
else:
    active_train_records = train_data
    active_dev_records = dev_data

TRAIN_CACHE = build_split_cache(active_train_records, MAX_CHUNKS_TRAIN, "train")
DEV_CACHE = build_split_cache(active_dev_records, MAX_CHUNKS_EVAL, "dev")
TEST_CACHE = build_split_cache(test_data, MAX_CHUNKS_EVAL, "test")


class CJPEDataset(Dataset):
    def __init__(self, cached_records):
        self.records = cached_records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        return {
            "id": rec["id"],
            "chunks": torch.from_numpy(rec["chunks"].astype(np.int64)),
            "lengths": torch.from_numpy(rec["lengths"].astype(np.int64)),
            "label": rec["label"],
        }


# all chunks of the batch are stacked into one tensor, doc_counts says which chunks belong to which case
def cjpe_collate_fn(batch):
    ids = [b["id"] for b in batch]
    chunk_tensors = [b["chunks"] for b in batch]
    len_tensors = [b["lengths"] for b in batch]
    doc_counts = [c.size(0) for c in chunk_tensors]

    flat_chunks = torch.cat(chunk_tensors, dim=0)
    flat_lengths = torch.cat(len_tensors, dim=0)
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)

    return {
        "ids": ids,
        "flat_chunks": flat_chunks,
        "flat_lengths": flat_lengths,
        "doc_counts": doc_counts,
        "labels": labels,
    }


train_dataset = CJPEDataset(TRAIN_CACHE)
dev_dataset = CJPEDataset(DEV_CACHE)
test_dataset = CJPEDataset(TEST_CACHE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=cjpe_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=cjpe_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=cjpe_collate_fn)

_sample_batch = next(iter(train_loader))
log(f"loaders ready | train batches {len(train_loader)}, dev {len(dev_loader)}, test {len(test_loader)}")
log(f"sample batch: {len(_sample_batch['ids'])} cases, flat chunks {tuple(_sample_batch['flat_chunks'].shape)}")


## Phase 1 model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return self.weight * (x * torch.rsqrt(variance + self.eps))


class SwiGLU(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.w1 = nn.Linear(in_features, hidden_features, bias=False)
        self.w2 = nn.Linear(in_features, hidden_features, bias=False)
        self.w3 = nn.Linear(hidden_features, in_features, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


_ROPE = {}


def _rope_tables(head_dim, device, max_len=2048):
    key = (head_dim, str(device))
    if key not in _ROPE:
        # rotary position tables
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
        t = torch.arange(max_len, device=device).float()
        emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
        _ROPE[key] = (emb.cos(), emb.sin())
    return _ROPE[key]


def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k):
    T, hd = q.size(-2), q.size(-1)
    cos, sin = _rope_tables(hd, q.device)
    cos = cos[:T].to(q.dtype)[None, None]
    sin = sin[:T].to(q.dtype)[None, None]
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


_MASK = {}


def causal_window_mask(T, window, device):
    key = (T, window, str(device))
    m = _MASK.get(key)
    if m is None:
        _MASK.clear()
        i = torch.arange(T, device=device)[:, None]
        j = torch.arange(T, device=device)[None, :]
        # look back only, and only `window` tokens
        m = (j <= i) & ((i - j) < window)
        _MASK[key] = m
    return m


def attend(q, k, v, window=None):
    T = q.size(-2)
    # global layer or short input: plain causal attention
    if window is None or window >= T:
        return F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=causal_window_mask(T, window, q.device))


def init_weights(model, n_layers):
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    model.apply(_init)
    # smaller init for the output projections
    for name, p in model.named_parameters():
        if name.endswith("o_proj.weight") or name.endswith("w3.weight"):
            nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * max(1, n_layers)))


# fewer k/v heads than q heads (grouped query)
class GQACausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, window=None):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.num_queries_per_kv = n_heads // n_kv_heads
        self.window = window
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k)
        # repeat k and v so every group of q heads gets a copy
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
        y = attend(q, k, v, self.window)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, window=None):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQACausalAttention(d_model, n_heads, n_kv_heads, window)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))


class _CausalLMBase(nn.Module):
    def __init__(self, vocab_size, d_model, layers, max_seq_len):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.use_grad_ckpt = True
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(layers)
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # tied weights
        self.lm_head.weight = self.tok_emb.weight
        init_weights(self, len(layers))

    # hidden states of every token, this is what the classifier pools over
    def forward_features(self, idx):
        x = self.tok_emb(idx)
        for layer in self.layers:
            # checkpoint only while training and only when the activations need gradients
            if self.use_grad_ckpt and self.training and x.requires_grad:
                x = grad_ckpt(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.final_norm(x)

    def forward(self, idx, labels=None):
        x = self.forward_features(idx)
        if labels is None:
            x = x[:, -1:, :]
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.reshape(-1))
        return SimpleNamespace(logits=logits, loss=loss)


class LegalEfficientTransformer(_CausalLMBase):
    def __init__(self, vocab_size, d_model=1024, n_heads=16, n_kv_heads=4, n_layers=16, d_ff=2816,
                 window=ATTN_WINDOW, global_every=4, max_seq_len=MAX_SEQ_LEN):
        # every global_every-th layer is global (window None), the rest use the sliding window
        layers = [EfficientDecoderBlock(d_model, n_heads, n_kv_heads, d_ff,
                                        window=None if (i + 1) % global_every == 0 else window)
                  for i in range(n_layers)]
        super().__init__(vocab_size, d_model, layers, max_seq_len)


# build the phase 1 model from the config saved in the checkpoint
base_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
raw_state = base_ck.get("state_dict", base_ck.get("model", base_ck))
# remove the torch.compile prefix if it is there
state = {k.replace("_orig_mod.", "", 1): v for k, v in raw_state.items()}

if "config" in base_ck:
    cfg_ = dict(base_ck["config"])
else:
    # old checkpoints without a config, read the sizes from the weights
    _d_model = state["tok_emb.weight"].shape[1]
    _d_ff = state["layers.0.ffn.w1.weight"].shape[0]
    _n_layers = max(int(k.split(".")[1]) for k in state if k.startswith("layers.")) + 1
    cfg_ = dict(vocab_size=VOCAB_SIZE, d_model=_d_model, n_heads=16 if _d_model == 1024 else 12,
                n_kv_heads=4, n_layers=_n_layers, d_ff=_d_ff)

base = LegalEfficientTransformer(**cfg_, window=ATTN_WINDOW, global_every=GLOBAL_EVERY)
missing, unexpected = base.load_state_dict(state, strict=False)
# lm_head is tied to tok_emb so it is not always in the file
missing = [k for k in missing if k != "lm_head.weight"]
assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
log(f"loaded {CKPT_PATH} (step {base_ck.get('step', 'N/A')}, val loss {base_ck.get('val_loss', 'N/A')})")
print("config:", cfg_)
del base_ck, raw_state
base = base.to(dev).eval()
assert base.tok_emb.weight.shape[0] == VOCAB_SIZE, "tokenizer vocab size does not match the checkpoint"
print(f"backbone parameters: {sum(p.numel() for p in base.parameters()) / 1e6:.2f}M")
print("layer windows:", [l.attn.window for l in base.layers])


# a clean copy of the phase 1 weights for every experiment, built on the cpu
def fresh_base():
    m = LegalEfficientTransformer(**cfg_, window=ATTN_WINDOW, global_every=GLOBAL_EVERY)
    m.load_state_dict(state, strict=False)
    return m


## Classifier

In [ ]:
# same idea as lora but the low rank update acts on the output of W, not on its input
class LoRMALinear(nn.Module):
    def __init__(self, base_linear, rank=8, alpha=16.0, dropout=0.05):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad_(False)
        out_features = base_linear.weight.size(0)
        self.A = nn.Parameter(torch.randn(rank, out_features) * 0.02)
        # B starts at zero so at the start it behaves exactly like phase 1
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = alpha / float(rank)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.base(x)
        delta = (self.drop(h) @ self.A.t()) @ self.B.t()
        return h + self.scale * delta


# frozen W plus a small trainable B @ A
class LoRALinear(nn.Module):
    def __init__(self, base_linear, rank=8, alpha=16.0, dropout=0.05):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad_(False)
        out_features, in_features = base_linear.weight.shape
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = alpha / float(rank)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # W x + scale * B A x
        return self.base(x) + self.scale * ((self.drop(x) @ self.A.t()) @ self.B.t())


# every 512 token chunk is pooled (mean, max, eos) into one vector, then an attention pool over the chunks gives the case vector
class HierarchicalCJPEClassifier(nn.Module):
    def __init__(self, backbone, num_classes=2, chunk_micro_batch=32):
        super().__init__()
        self.backbone = backbone
        self.d_model = backbone.tok_emb.embedding_dim
        self.chunk_micro_batch = chunk_micro_batch

        self.chunk_proj = nn.Sequential(
            nn.Linear(3 * self.d_model, self.d_model),
            nn.LayerNorm(self.d_model),
            nn.GELU(),
            nn.Dropout(0.15),
        )

        # gated attention over the chunks of one case
        self.attn_V = nn.Linear(self.d_model, self.d_model // 2)
        self.attn_U = nn.Linear(self.d_model, self.d_model // 2)
        self.attn_w = nn.Linear(self.d_model // 2, 1, bias=False)

        self.classifier = nn.Sequential(
            nn.Linear(2 * self.d_model, self.d_model // 2),
            nn.LayerNorm(self.d_model // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(self.d_model // 2, num_classes),
        )

    def _encode_chunks(self, flat_chunks, flat_lengths):
        total_c, T = flat_chunks.shape
        device = flat_chunks.device
        chunk_embs = []

        for s in range(0, total_c, self.chunk_micro_batch):
            sub_chunks = flat_chunks[s: s + self.chunk_micro_batch]
            sub_lens = flat_lengths[s: s + self.chunk_micro_batch]
            hs = self.backbone.forward_features(sub_chunks)

            # padding is not part of the pooling
            arange_t = torch.arange(T, device=device).unsqueeze(0)
            valid_mask = (arange_t < sub_lens.unsqueeze(1)).unsqueeze(-1)
            mask_f = valid_mask.to(hs.dtype)

            mean_p = (hs * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
            masked_hs = hs.masked_fill(~valid_mask, -1e4)
            max_p = masked_hs.max(dim=1).values
            last_idx = (sub_lens - 1).clamp(min=0)
            eos_p = hs[torch.arange(hs.size(0), device=device), last_idx]

            combined = torch.cat([mean_p, max_p, eos_p], dim=-1)
            chunk_embs.append(self.chunk_proj(combined))

        return torch.cat(chunk_embs, dim=0)

    def forward(self, flat_chunks, flat_lengths, doc_counts):
        chunk_vecs = self._encode_chunks(flat_chunks, flat_lengths)
        doc_splits = torch.split(chunk_vecs, doc_counts, dim=0)

        case_logits = []
        for doc_chunks in doc_splits:
            gates = torch.tanh(self.attn_V(doc_chunks)) * torch.sigmoid(self.attn_U(doc_chunks))
            scores = self.attn_w(gates)
            weights = F.softmax(scores, dim=0)
            attn_pooled = (weights * doc_chunks).sum(dim=0)
            mean_pooled = doc_chunks.mean(dim=0)

            case_vec = torch.cat([attn_pooled, mean_pooled], dim=-1)
            case_logits.append(self.classifier(case_vec))

        return torch.stack(case_logits, dim=0)


# freeze the phase 1 weights, then open up whatever the method needs
def make_model(method):
    backbone = fresh_base()

    for p in backbone.parameters():
        p.requires_grad_(False)

    if method == "lora":
        wrapped = 0
        for blk in backbone.layers:
            for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(blk.attn, name, LoRALinear(getattr(blk.attn, name), RANK, ALPHA, LORA_DROPOUT))
                wrapped += 1
            for name in ("w1", "w2", "w3"):
                setattr(blk.ffn, name, LoRALinear(getattr(blk.ffn, name), RANK, ALPHA, LORA_DROPOUT))
                wrapped += 1
        for p in backbone.final_norm.parameters():
            p.requires_grad_(True)
        print(f"wrapped {wrapped} attention + ffn linears with lora (r={RANK}, alpha={ALPHA})")

    elif method == "lorma":
        wrapped = 0
        for blk in backbone.layers:
            for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(blk.attn, name, LoRMALinear(getattr(blk.attn, name), RANK, ALPHA, LORA_DROPOUT))
                wrapped += 1
            for p in blk.attn_norm.parameters():
                p.requires_grad_(True)
        for p in backbone.final_norm.parameters():
            p.requires_grad_(True)
        print(f"wrapped {wrapped} attention linears with lorma (r={RANK}, alpha={ALPHA})")

    elif method == "full":
        for p in backbone.parameters():
            p.requires_grad_(True)
        print("every backbone weight is trainable")

    elif method == "distillation":
        # student = every second phase 1 layer, lora on the attention and the last 2 blocks trainable
        total_l = len(backbone.layers)
        even_indices = list(range(0, total_l, 2))[:STUDENT_LAYERS]
        backbone.layers = nn.ModuleList([backbone.layers[i] for i in even_indices])
        for blk in backbone.layers:
            for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(blk.attn, name, LoRALinear(getattr(blk.attn, name), RANK, ALPHA, LORA_DROPOUT))
        for blk in backbone.layers[-2:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in backbone.final_norm.parameters():
            p.requires_grad_(True)
        print(f"student has {len(backbone.layers)} layers, taken from phase 1 layers {even_indices}")

    else:
        raise ValueError(f"unknown method: {method}")

    model = HierarchicalCJPEClassifier(backbone, num_classes=2, chunk_micro_batch=CHUNK_MICRO_BATCH).to(dev)
    tr_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot_params = sum(p.numel() for p in model.parameters())
    print(f"[{method.upper()}] trainable {tr_params / 1e6:.2f}M / total {tot_params / 1e6:.2f}M ({tr_params / tot_params:.1%})")
    return model


In [ ]:
# check the phase 1 weights and the tokenizer match, lm loss on a train case should be low
sample = " ".join(train_data[0]["text"].split()[:600])
ids0 = torch.tensor([tokenizer.encode(sample)[:CHUNK_LEN]], device=dev)
with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
    lg = base.lm_head(base.forward_features(ids0)).float()
    lm_loss = F.cross_entropy(lg[0, :-1], ids0[0, 1:]).item()
print(f"lm loss on a train case: {lm_loss:.3f}")
assert lm_loss < 6.0, "lm loss too high, weights and tokenizer probably do not match"
del lg


## Evaluation and submission

In [ ]:
# logits for every case of a loader, in loader order
@torch.no_grad()
def get_split_logits(model, loader, desc=None):
    model.eval()
    all_ids, all_logits, all_labels = [], [], []
    total_b = len(loader)
    t0 = time.time()
    for b_idx, batch in enumerate(loader, 1):
        flat_chunks = batch["flat_chunks"].to(dev, non_blocking=True)
        flat_lengths = batch["flat_lengths"].to(dev, non_blocking=True)
        doc_counts = batch["doc_counts"]
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            logits = model(flat_chunks, flat_lengths, doc_counts)
        all_ids.extend(batch["ids"])
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(batch["labels"].numpy())
        if desc and (b_idx % 200 == 0 or b_idx == total_b):
            log(f"  [{desc}] {b_idx}/{total_b} batches ({b_idx / total_b:.0%}) in {time.time() - t0:.1f}s")
    return all_ids, np.concatenate(all_logits, axis=0), np.concatenate(all_labels, axis=0)


# dev loss and macro scores, the decision threshold on p(class 1) is tuned on dev
def evaluate_dev(model, loader, class_weights=CLASS_WEIGHTS, tune_threshold=True, verbose=False):
    _, logits, labels = get_split_logits(model, loader, desc="dev eval" if verbose else None)
    logits_t = torch.from_numpy(logits).to(dev)
    labels_t = torch.from_numpy(labels).to(dev)
    val_loss = F.cross_entropy(logits_t, labels_t, weight=class_weights).item()

    probs_1 = F.softmax(logits_t, dim=-1)[:, 1].cpu().numpy()

    best_tau = 0.50
    best_f1 = -1.0
    if tune_threshold:
        for tau in np.linspace(0.20, 0.80, 61):
            preds = (probs_1 >= tau).astype(int)
            f1 = f1_score(labels, preds, average="macro", zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_tau = float(tau)
    else:
        preds = (probs_1 >= 0.50).astype(int)
        best_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    final_preds = (probs_1 >= best_tau).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(labels, final_preds, average="macro", zero_division=0)

    if verbose:
        print(f"\ndev classification report (threshold {best_tau:.2f})", flush=True)
        print(classification_report(labels, final_preds, target_names=["Class 0 (Rejected)", "Class 1 (Accepted)"], digits=4, zero_division=0), flush=True)

    return {
        "val_loss": val_loss,
        "macro_f1": float(f1),
        "precision": float(p),
        "recall": float(r),
        "threshold": best_tau,
        "dev_logits": logits,
    }


# runs the test set, checks ids and labels, writes prediction.json and zips it
def generate_submission(model, loader, test_records, output_zip_path, threshold=0.50):
    log(f"running inference on test ({len(test_records)} cases) -> {Path(output_zip_path).name}")
    pred_ids, test_logits, _ = get_split_logits(model, loader, desc="test inference")
    probs_1 = F.softmax(torch.from_numpy(test_logits), dim=-1)[:, 1].numpy()
    pred_labels = (probs_1 >= threshold).astype(int).tolist()

    id_to_pred = {cid: int(p) for cid, p in zip(pred_ids, pred_labels)}
    expected_ids = [r["id"] for r in test_records]

    submission_list = []
    for cid in expected_ids:
        assert cid in id_to_pred, f"missing prediction for test id {cid}"
        pred_val = int(id_to_pred[cid])
        assert pred_val in (0, 1), f"invalid prediction {pred_val} for id {cid}"
        submission_list.append({"id": str(cid), "pred": pred_val})

    assert len(submission_list) == len(test_records), "prediction count mismatch"
    assert [x["id"] for x in submission_list] == expected_ids, "order / id mismatch with test.json"

    output_zip_path = Path(output_zip_path)
    pred_json_path = OUT_DIR / f"prediction_{output_zip_path.stem}.json"
    with open(pred_json_path, "w", encoding="utf-8") as fh:
        json.dump(submission_list, fh, indent=2, ensure_ascii=False)

    # prediction.json has to be at the root of the zip
    with zipfile.ZipFile(output_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_json_path, arcname="prediction.json")

    with zipfile.ZipFile(output_zip_path, "r") as zf:
        assert zf.namelist() == ["prediction.json"], f"zip root error: {zf.namelist()}"
        loaded_back = json.loads(zf.read("prediction.json").decode("utf-8"))
        assert len(loaded_back) == len(test_records)

    dist = Counter(x["pred"] for x in submission_list)
    log(f"wrote {output_zip_path} ({output_zip_path.stat().st_size / 1024:.1f} KB)")
    log(f"  {len(submission_list)} cases | class 0: {dist[0]} ({dist[0] / len(submission_list):.1%}) | class 1: {dist[1]} ({dist[1] / len(submission_list):.1%})")
    try:
        display(FileLink(str(output_zip_path.relative_to(WORK_DIR))))
    except Exception:
        pass
    return submission_list


## Training loop

In [ ]:
# best non distillation run so far, its checkpoint is the teacher for the student
BEST_TEACHER = {"f1": -1.0, "exp_num": None, "method": None, "ckpt_path": None}
EXPERIMENT_RESULTS = {}


# soft logits of the best teacher on the train set, computed once before the student starts
def teacher_soft_logits():
    assert BEST_TEACHER["ckpt_path"] is not None, "no teacher checkpoint found for distillation"
    log(f"loading best teacher ({BEST_TEACHER['method']}, dev macro-F1 {BEST_TEACHER['f1']:.4f}) for the soft targets")
    teacher_model = make_model(BEST_TEACHER["method"])
    t_ck = torch.load(BEST_TEACHER["ckpt_path"], map_location=dev, weights_only=False)
    teacher_model.load_state_dict(t_ck["state_dict"])
    eval_train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=cjpe_collate_fn)
    t_ids, t_lg, _ = get_split_logits(teacher_model, eval_train_loader, desc="teacher soft logits")
    teacher_map = {cid: lg for cid, lg in zip(t_ids, t_lg)}
    del teacher_model, t_ck
    gc.collect(); torch.cuda.empty_cache()
    log(f"cached soft logits for {len(teacher_map)} train cases, teacher is out of the gpu")
    return teacher_map


# head and backbone get different lrs, 1d params (norms) skip the weight decay
def make_optim(model, backbone_lr, head_lr, epochs):
    head_params, bb_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("backbone."):
            bb_params.append(p)
        else:
            head_params.append(p)

    groups = [{"params": head_params, "lr": head_lr, "weight_decay": WEIGHT_DECAY}]
    decay_p = [p for p in bb_params if p.ndim >= 2]
    nodecay_p = [p for p in bb_params if p.ndim < 2]
    if decay_p:
        groups.append({"params": decay_p, "lr": backbone_lr, "weight_decay": WEIGHT_DECAY})
    if nodecay_p:
        groups.append({"params": nodecay_p, "lr": backbone_lr, "weight_decay": 0.0})
    optimizer = torch.optim.AdamW(groups)

    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    total_steps = max(1, steps_per_epoch * epochs)
    warmup_steps = max(1, int(WARMUP_FRAC * total_steps))
    # linear warmup, then cosine decay down to 5%
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda s: float(s + 1) / warmup_steps if s < warmup_steps
        else max(0.05, 0.5 * (1.0 + math.cos(math.pi * float(s - warmup_steps) / max(1, total_steps - warmup_steps)))))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    return optimizer, scheduler, scaler


# keeps the checkpoint with the best dev macro-F1 and loads it back at the end
# teacher_map given = distillation (ce on the labels + kl against the teacher logits)
def run_training(model, optimizer, scheduler, scaler, tag, epochs, ckpt_file, teacher_map=None):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    best_f1 = -1.0
    exp_t0 = time.time()
    num_batches = len(train_loader)

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        ep_t0 = time.time()

        log(f"[{tag}] epoch {epoch}/{epochs} ({num_batches} batches, batch size {BATCH_SIZE}, grad accum {GRAD_ACCUM})")
        for b_idx, batch in enumerate(train_loader, 1):
            flat_chunks = batch["flat_chunks"].to(dev, non_blocking=True)
            flat_lengths = batch["flat_lengths"].to(dev, non_blocking=True)
            doc_counts = batch["doc_counts"]
            labels = batch["labels"].to(dev, non_blocking=True)

            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                logits = model(flat_chunks, flat_lengths, doc_counts)
                ce_loss = F.cross_entropy(logits.float(), labels, weight=CLASS_WEIGHTS, label_smoothing=LABEL_SMOOTHING)

                if teacher_map is not None:
                    t_logits = torch.tensor(
                        np.stack([teacher_map[cid] for cid in batch["ids"]], axis=0),
                        dtype=torch.float32, device=dev)
                    kd_loss = F.kl_div(
                        F.log_softmax(logits.float() / KD_TEMP, dim=-1),
                        F.softmax(t_logits / KD_TEMP, dim=-1),
                        reduction="batchmean") * (KD_TEMP ** 2)
                    loss = KD_ALPHA * ce_loss + (1.0 - KD_ALPHA) * kd_loss
                else:
                    loss = ce_loss

                loss_scaled = loss / GRAD_ACCUM

            scaler.scale(loss_scaled).backward()
            running_loss += float(loss.detach())

            # step every GRAD_ACCUM batches and on the last one
            if b_idx % GRAD_ACCUM == 0 or b_idx == num_batches:
                # unscale first so the clipping sees the real gradients
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            # progress line so the logs never look frozen
            if b_idx == 1 or b_idx == 10 or b_idx % LOG_EVERY == 0 or b_idx == num_batches:
                elapsed_ep = time.time() - ep_t0
                speed = b_idx * BATCH_SIZE / max(elapsed_ep, 1e-3)
                eta_min = ((num_batches - b_idx) * (elapsed_ep / b_idx)) / 60.0
                log(f"[{tag} | ep {epoch}/{epochs}] batch {b_idx:4d}/{num_batches} ({b_idx / num_batches:5.1%}) | "
                    f"loss {running_loss / b_idx:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | "
                    f"{speed:.1f} cases/s | {elapsed_ep / 60:.1f}m | eta {eta_min:.1f}m")

        # dev score after every epoch
        metrics = evaluate_dev(model, dev_loader, CLASS_WEIGHTS, tune_threshold=True, verbose=False)
        log(f"[{tag} | ep {epoch}/{epochs}] train loss {running_loss / num_batches:.4f} | dev loss {metrics['val_loss']:.4f} | "
            f"dev macro F1 {metrics['macro_f1']:.4f} (P {metrics['precision']:.3f}, R {metrics['recall']:.3f}, tau {metrics['threshold']:.2f}) | "
            f"{(time.time() - exp_t0) / 60:.1f} min")

        if metrics["macro_f1"] > best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({"state_dict": model.state_dict(), "macro_f1": best_f1, "threshold": metrics["threshold"]}, ckpt_file)
            log(f"  saved new best checkpoint {Path(ckpt_file).name} (macro F1 {best_f1:.4f})")

    # go back to the best epoch
    best_ck = torch.load(ckpt_file, map_location=dev, weights_only=False)
    model.load_state_dict(best_ck["state_dict"])
    return best_f1


## Pre-flight check

In [ ]:
# one tiny forward / backward / dev eval / zip with lorma before the real runs, so a broken path fails in seconds
if RUN_PREFLIGHT:
    log("running the pre-flight check")
    _smoke_train = DataLoader(CJPEDataset(TRAIN_CACHE[:16]), batch_size=4, shuffle=False, collate_fn=cjpe_collate_fn)
    _smoke_dev = DataLoader(CJPEDataset(DEV_CACHE[:16]), batch_size=4, shuffle=False, collate_fn=cjpe_collate_fn)
    _smoke_test = DataLoader(CJPEDataset(TEST_CACHE[:16]), batch_size=4, shuffle=False, collate_fn=cjpe_collate_fn)

    _smoke_model = make_model("lorma")
    _smoke_opt = torch.optim.AdamW([p for p in _smoke_model.parameters() if p.requires_grad], lr=1e-4)
    _smoke_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    _smoke_model.train()
    for _b in _smoke_train:
        _fc = _b["flat_chunks"].to(dev)
        _fl = _b["flat_lengths"].to(dev)
        _lb = _b["labels"].to(dev)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            _lg = _smoke_model(_fc, _fl, _b["doc_counts"])
            _loss = F.cross_entropy(_lg.float(), _lb, weight=CLASS_WEIGHTS)
        _smoke_opt.zero_grad(set_to_none=True)
        _smoke_scaler.scale(_loss).backward()
        _smoke_scaler.step(_smoke_opt)
        _smoke_scaler.update()
        break

    _smoke_metrics = evaluate_dev(_smoke_model, _smoke_dev, CLASS_WEIGHTS, tune_threshold=True, verbose=False)
    _smoke_zip = OUT_DIR / "smoke_test_check.zip"
    generate_submission(_smoke_model, _smoke_test, test_data[:16], _smoke_zip, threshold=_smoke_metrics["threshold"])
    if _smoke_zip.exists():
        _smoke_zip.unlink()

    del _smoke_model, _smoke_opt, _smoke_scaler, _smoke_train, _smoke_dev, _smoke_test
    gc.collect(); torch.cuda.empty_cache()
    log("pre-flight check passed")


## Train

In [ ]:
del base
gc.collect(); torch.cuda.empty_cache()

for exp in EXPERIMENTS:
    num, method = exp["num"], exp["method"]
    epochs = 1 if SMOKE else exp["epochs"]
    seed_everything(SEED + num)
    log(f"===== exp {num}/{len(EXPERIMENTS)}: {method} (epochs {epochs}, backbone lr {exp['backbone_lr']}, head lr {exp['head_lr']}) =====")

    # the student learns from the soft logits of the best teacher so far
    teacher_map = teacher_soft_logits() if exp.get("kd") else None

    model = make_model(method)
    optimizer, scheduler, scaler = make_optim(model, exp["backbone_lr"], exp["head_lr"], epochs)
    ckpt_file = OUT_DIR / f"best_exp{num}_{method}.pt"
    run_training(model, optimizer, scheduler, scaler, f"exp{num} {method}", epochs, ckpt_file, teacher_map)

    final_dev = evaluate_dev(model, dev_loader, CLASS_WEIGHTS, tune_threshold=True, verbose=True)
    best_f1, best_tau = final_dev["macro_f1"], final_dev["threshold"]

    if method != "distillation" and best_f1 > BEST_TEACHER["f1"]:
        BEST_TEACHER.update({"f1": best_f1, "exp_num": num, "method": method, "ckpt_path": str(ckpt_file)})
        log(f"{method} is the best teacher so far (dev macro F1 {best_f1:.4f})")

    zip_path = WORK_DIR / f"submission_exp{num}_{method}.zip"
    generate_submission(model, test_loader, test_data, zip_path, threshold=best_tau)

    EXPERIMENT_RESULTS[f"Exp {num}: {method}"] = {
        "macro_f1": round(best_f1, 4),
        "precision": round(final_dev["precision"], 4),
        "recall": round(final_dev["recall"], 4),
        "threshold": round(best_tau, 2),
        "zip_path": str(zip_path),
    }

    del model, optimizer, scheduler, scaler, teacher_map
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "=" * 95, flush=True)
print(f"{'Experiment':<26} | {'Macro-F1':<10} | {'Precision':<10} | {'Recall':<10} | {'Tau':<6} | {'Submission File'}", flush=True)
print("-" * 95, flush=True)
for exp_name, res in EXPERIMENT_RESULTS.items():
    print(f"{exp_name:<26} | {res['macro_f1']:<10.4f} | {res['precision']:<10.4f} | {res['recall']:<10.4f} | {res['threshold']:<6.2f} | {res['zip_path']}", flush=True)
print("=" * 95, flush=True)

with open(WORK_DIR / "phase4_cjpe_summary.json", "w", encoding="utf-8") as fh:
    json.dump(EXPERIMENT_RESULTS, fh, indent=2)
